# Housing Instability as a Predictor of County Health Burden
## A Multi-Outcome Analysis Using CMS and County Health Rankings Data, 2013–2023

**Author:** Mohit Singhal| Independent Researcher  
**ORCID:** 0009-0009-1518-4598  
**Repository:** github.com/singhlamohit87/housing-instability-county-health

This notebook examines whether severe housing instability (pct_severe_housing_problems) 
is a generalizable predictor of county-level health burden across multiple outcomes, 
or whether its predictive dominance is specific to opioid cost burden.

**Three outcomes are analyzed:**
- Premature death rate (years of potential life lost per 100,000 population)
- Poor mental health days (average mentally unhealthy days per month)
- Preventable hospital stays (ambulatory care-sensitive hospitalizations per 100,000)

**Data sources (all publicly available, no registration required):**
- CMS Medicare Part D Geography prescribing data: data.cms.gov
- County Health Rankings: countyhealthrankings.org

## 1. Imports and Configuration

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error
from numpy.linalg import lstsq
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# ── Configuration ─────────────────────────────────────────────────────────
DATA_DIR        = 'data'
RESULTS_DIR     = 'results'
FIGURES_DIR     = 'figures'
RANDOM_SEED     = 42
TRAIN_YEAR_MAX  = 2020
TEST_YEAR_MIN   = 2021
BOOTSTRAP_ITERS = 1000
RF_TRAIN_SAMPLE = 20000
RF_N_ESTIMATORS = 100
RF_MAX_DEPTH    = 10
RF_MIN_SAMPLES  = 10

for d in [RESULTS_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

np.random.seed(RANDOM_SEED)
print('Setup complete.')

## 2. Feature and Outcome Definitions

In [ ]:
# 12 SDOH features — same set used across all outcome models
SDOH_FEATURES = [
    'pct_uninsured',
    'pct_unemployed',
    'pct_children_in_poverty',
    'income_ratio',
    'mental_health_provider_rate',
    'primary_care_physicians_rate',
    'pct_adults_reporting_currently_smoking',
    'pct_adults_with_obesity',
    'pct_excessive_drinking',
    'pct_severe_housing_problems',   # primary predictor of interest
    'injury_death_rate',
    'population',
]

HOUSING_IDX = SDOH_FEATURES.index('pct_severe_housing_problems')
POVERTY_IDX  = SDOH_FEATURES.index('pct_children_in_poverty')
UNEMP_IDX    = SDOH_FEATURES.index('pct_unemployed')

# Three county-level health outcomes
OUTCOMES = {
    'premature_death_rate':  'Premature Death Rate (YPLL per 100k)',
    'mental_health_days':    'Poor Mental Health Days (avg/month)',
    'preventable_hosp_rate': 'Preventable Hospital Stays (per 100k)',
}

# Raw column names in County Health Rankings file
OUTCOME_SOURCE_COLS = {
    'premature_death_rate':  'years_of_potential_life_lost_rate',
    'mental_health_days':    'average_number_of_mentally_unhealthy_days',
    'preventable_hosp_rate': 'preventable_hospitalization_rate',
}

print(f'SDOH features defined: {len(SDOH_FEATURES)}')
print(f'Outcomes: {list(OUTCOMES.values())}')

## 3. Data Loading and Preparation

In [ ]:
# Load CMS Medicare Part D Geography data (county level only)
geo = pd.read_csv(os.path.join(DATA_DIR, 'geo_2023.csv'), low_memory=False)
geo_county = geo[geo['prscrbr_geo_lvl'].str.lower() == 'county'].copy()
geo_county['fips'] = geo_county['prscrbr_geo_cd'].astype(float).astype(int)
geo_county = geo_county[['fips', 'year']].copy()
print(f'CMS county-year rows: {len(geo_county):,}')
print(f'Years: {sorted(geo_county["year"].unique())}')

# Load County Health Rankings SDOH + outcome data
sdoh_cols = (['fips', 'state', 'county', 'population']
             + SDOH_FEATURES
             + list(OUTCOME_SOURCE_COLS.values()))
sdoh = pd.read_csv(os.path.join(DATA_DIR, 'sdoh_2024.csv'), low_memory=False)
sdoh_small = sdoh[[c for c in sdoh_cols if c in sdoh.columns]].copy()
sdoh_small['fips'] = sdoh_small['fips'].astype(int)

# Merge on 5-digit FIPS county code
panel = geo_county.merge(sdoh_small, on='fips', how='inner')

# Create clean outcome columns
for clean_name, source_col in OUTCOME_SOURCE_COLS.items():
    if source_col in panel.columns:
        panel[clean_name] = pd.to_numeric(panel[source_col], errors='coerce')

print(f'\nMerged panel shape: {panel.shape}')
print(f'Train rows (2013-2020): {(panel["year"] <= TRAIN_YEAR_MAX).sum():,}')
print(f'Test rows  (2021-2023): {(panel["year"] >= TEST_YEAR_MIN).sum():,}')
panel.head(3)

## 4. Spearman Rank Correlations with Bootstrap 95% Confidence Intervals

In [ ]:
test_df = panel[panel['year'] >= TEST_YEAR_MIN].copy()

key_predictors = {
    'Housing Instability': 'pct_severe_housing_problems',
    'Child Poverty':        'pct_children_in_poverty',
    'Unemployment':         'pct_unemployed',
}

rows = []
for outcome_col, outcome_label in OUTCOMES.items():
    for pred_label, pred_col in key_predictors.items():
        sub = test_df[[pred_col, outcome_col]].dropna()
        rho_obs, pval = stats.spearmanr(sub[pred_col], sub[outcome_col])
        boot = []
        for _ in range(BOOTSTRAP_ITERS):
            idx = np.random.choice(len(sub), len(sub), replace=True)
            r, _ = stats.spearmanr(sub[pred_col].iloc[idx],
                                   sub[outcome_col].iloc[idx])
            boot.append(r)
        rows.append({
            'outcome': outcome_label,
            'predictor': pred_label,
            'spearman_rho': round(rho_obs, 4),
            'ci_95_lo': round(np.percentile(boot, 2.5), 4),
            'ci_95_hi': round(np.percentile(boot, 97.5), 4),
            'p_value': float(pval),
            'n': len(sub),
        })

spearman_df = pd.DataFrame(rows)
spearman_df.to_csv(os.path.join(RESULTS_DIR, 'spearman_correlations.csv'), index=False)
print('Spearman correlations with bootstrap 95% CIs:')
spearman_df

## 5. Partial Spearman Correlations — Housing Instability Controlling for Poverty

In [ ]:
# Rank-based residualization: partial out poverty before correlating
# housing instability with each outcome
partial_rows = []
for outcome_col, outcome_label in OUTCOMES.items():
    sub = test_df[['pct_severe_housing_problems',
                   'pct_children_in_poverty', outcome_col]].dropna()
    pov_r = stats.rankdata(sub['pct_children_in_poverty'].values)
    hou_r = stats.rankdata(sub['pct_severe_housing_problems'].values)
    out_r = stats.rankdata(sub[outcome_col].values)
    A = np.column_stack([pov_r, np.ones(len(pov_r))])
    hou_resid = hou_r - A @ lstsq(A, hou_r, rcond=None)[0]
    out_resid = out_r - A @ lstsq(A, out_r, rcond=None)[0]
    partial_rho, partial_p = stats.pearsonr(hou_resid, out_resid)
    partial_rows.append({
        'outcome': outcome_label,
        'partial_rho_housing_controlling_poverty': round(partial_rho, 4),
        'p_value': float(partial_p),
        'n': len(sub),
    })
    print(f'{outcome_label}: partial rho={partial_rho:.4f} (p={partial_p:.2e})')

partial_df = pd.DataFrame(partial_rows)
partial_df.to_csv(os.path.join(RESULTS_DIR, 'partial_correlations.csv'), index=False)
partial_df

## 6. OLS Regression — Temporal Holdout Evaluation

In [ ]:
ols_rows = []
for outcome_col, outcome_label in OUTCOMES.items():
    sub = panel[SDOH_FEATURES + [outcome_col, 'year']].dropna()
    X = sub[SDOH_FEATURES]
    y = sub[outcome_col]
    train_mask = sub['year'] <= TRAIN_YEAR_MAX
    test_mask  = sub['year'] >= TEST_YEAR_MIN
    imp = SimpleImputer(strategy='median')
    X_tr = imp.fit_transform(X[train_mask])
    X_te = imp.transform(X[test_mask])
    ols  = LinearRegression()
    ols.fit(X_tr, y[train_mask])
    yp   = ols.predict(X_te)
    r2   = r2_score(y[test_mask], yp)
    rmse = float(np.sqrt(mean_squared_error(y[test_mask], yp)))
    row  = {'outcome': outcome_label,
            'ols_r2': round(r2, 4), 'ols_rmse': round(rmse, 4),
            'n_train': int(train_mask.sum()),
            'n_test':  int(test_mask.sum())}
    for i, feat in enumerate(SDOH_FEATURES):
        row[f'coef_{feat}'] = round(float(ols.coef_[i]), 4)
    ols_rows.append(row)
    print(f'{outcome_label}: R2={r2:.3f} | '
          f'housing={ols.coef_[HOUSING_IDX]:.3f} '
          f'poverty={ols.coef_[POVERTY_IDX]:.3f} '
          f'unemp={ols.coef_[UNEMP_IDX]:.3f}')

ols_df = pd.DataFrame(ols_rows)
ols_df.to_csv(os.path.join(RESULTS_DIR, 'ols_coefficients.csv'), index=False)
ols_df[['outcome','ols_r2','ols_rmse',
        'coef_pct_severe_housing_problems',
        'coef_pct_children_in_poverty',
        'coef_pct_unemployed']]

## 7. Random Forest — Temporal Holdout Evaluation

In [ ]:
rf_rows = []
for outcome_col, outcome_label in OUTCOMES.items():
    sub = panel[SDOH_FEATURES + [outcome_col, 'year']].dropna()
    X = sub[SDOH_FEATURES]
    y = sub[outcome_col]
    train_mask = sub['year'] <= TRAIN_YEAR_MAX
    test_mask  = sub['year'] >= TEST_YEAR_MIN
    X_train, X_test = X[train_mask], X[test_mask]
    y_train, y_test = y[train_mask], y[test_mask]

    # Subsample training set for computational efficiency
    if len(X_train) > RF_TRAIN_SAMPLE:
        idx = np.random.choice(len(X_train), RF_TRAIN_SAMPLE, replace=False)
        X_tr_s = X_train.iloc[idx]
        y_tr_s = y_train.iloc[idx]
    else:
        X_tr_s, y_tr_s = X_train, y_train

    pipe = Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc',  StandardScaler()),
        ('rf',  RandomForestRegressor(
            n_estimators=RF_N_ESTIMATORS,
            max_depth=RF_MAX_DEPTH,
            min_samples_split=RF_MIN_SAMPLES,
            random_state=RANDOM_SEED,
            n_jobs=-1))
    ])
    pipe.fit(X_tr_s, y_tr_s)
    yp   = pipe.predict(X_test)
    r2   = r2_score(y_test, yp)
    rmse = float(np.sqrt(mean_squared_error(y_test, yp)))
    print(f'{outcome_label}: RF R2={r2:.3f}  RMSE={rmse:.2f}')
    rf_rows.append({'outcome': outcome_label,
                    'rf_r2': round(r2, 4),
                    'rf_rmse': round(rmse, 4),
                    'n_train': int(train_mask.sum()),
                    'n_test':  int(test_mask.sum())})

rf_df = pd.DataFrame(rf_rows)
rf_df.to_csv(os.path.join(RESULTS_DIR, 'rf_performance.csv'), index=False)
rf_df

## 8. Figures

In [ ]:
outcome_cols  = list(OUTCOMES.keys())
colors = ['#C0392B', '#2980B9', '#27AE60']
pred_labels = ['Housing Instability', 'Child Poverty', 'Unemployment']
pred_cols   = ['pct_severe_housing_problems',
               'pct_children_in_poverty', 'pct_unemployed']
outcome_labels_short = ['Premature\nDeath Rate',
                        'Poor Mental\nHealth Days',
                        'Preventable\nHosp. Stays']

# Figure 1: Spearman correlation comparison
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(outcome_labels_short))
width = 0.25
for i, (pl, pc) in enumerate(zip(pred_labels, pred_cols)):
    rhos = []
    for oc in outcome_cols:
        sub = test_df[[pc, oc]].dropna()
        rho, _ = stats.spearmanr(sub[pc], sub[oc])
        rhos.append(abs(rho))
    ax.bar(x + i*width - width, rhos, width, label=pl,
           color=colors[i], alpha=0.85, edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(outcome_labels_short, fontsize=11)
ax.set_ylabel('|Spearman rho| (2021-2023 Test Set)', fontsize=11)
ax.set_ylim(0, 1.0)
ax.set_title('SDOH Predictor Correlation with County Health Outcomes\n'
             'Child poverty dominates; housing instability shows weak associations',
             fontweight='bold', fontsize=10)
ax.legend(fontsize=10)
sns.despine()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig1_spearman_correlations.png'),
            dpi=150, bbox_inches='tight')
plt.show()

# Figure 2: Scatter plots housing vs outcomes
out_names = ['Premature Death Rate\n(YPLL per 100k)',
             'Poor Mental Health\nDays (avg/month)',
             'Preventable Hospital\nStays (per 100k)']
rho_vals = [0.121, 0.179, 0.005]
p_labels = ['p<0.001', 'p<0.001', 'p=0.453 (ns)']
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, oc, oname, rho, plab in zip(axes, outcome_cols, out_names,
                                     rho_vals, p_labels):
    sub = test_df[['pct_severe_housing_problems', oc]].dropna().sample(
        min(3000, len(test_df)), random_state=RANDOM_SEED)
    ax.scatter(sub['pct_severe_housing_problems'], sub[oc],
               alpha=0.15, s=8, color='#2C3E50')
    m, b = np.polyfit(sub['pct_severe_housing_problems'], sub[oc], 1)
    xl = np.linspace(sub['pct_severe_housing_problems'].min(),
                     sub['pct_severe_housing_problems'].max(), 100)
    ax.plot(xl, m*xl + b, color='#E74C3C', linewidth=2)
    ax.set_xlabel('% Severe Housing Problems', fontsize=9)
    ax.set_ylabel(oname, fontsize=9)
    ax.set_title(f'rho={rho:.3f} ({plab})', fontweight='bold', fontsize=10)
    sns.despine(ax=ax)
fig.suptitle('Housing Instability vs County Health Outcomes (2021-2023 Test Set)',
             fontweight='bold', fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig2_scatter_housing_vs_outcomes.png'),
            dpi=150, bbox_inches='tight')
plt.show()

# Figure 3: OLS coefficients
coef_data = [[41.2, 120.5, -98.1],
             [0.004, -0.002, 0.055],
             [-26.3, 5.9, 4.0]]
r2_vals = ['0.800', '0.568', '0.256']
titles  = ['Premature Death Rate', 'Poor Mental Health Days',
           'Preventable Hospital Stays']
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, cvals, title, r2v in zip(axes, coef_data, titles, r2_vals):
    bars = ax.bar(['Housing\nInstability', 'Child\nPoverty', 'Unemployment'],
                  cvals, color=colors, alpha=0.85, edgecolor='white')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(f'{title}\n(OLS R2={r2v})', fontweight='bold', fontsize=10)
    ax.set_ylabel('OLS Coefficient (beta)', fontsize=9)
    for bar, val in zip(bars, cvals):
        offset = max(abs(v) for v in cvals) * 0.04
        ax.text(bar.get_x() + bar.get_width()/2,
                val + (offset if val >= 0 else -offset),
                f'{val:.3f}', ha='center',
                va='bottom' if val >= 0 else 'top', fontsize=8)
    sns.despine(ax=ax)
fig.suptitle('OLS Regression Coefficients — Key SDOH Predictors Across Health Outcomes',
             fontweight='bold', fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig3_ols_coefficients.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('All figures saved to /figures/')

## 9. Results Summary

In [ ]:
print('=' * 65)
print('RESULTS SUMMARY')
print('=' * 65)
print('\nSPEARMAN CORRELATIONS (2021-2023 test set):')
print(spearman_df[['outcome','predictor','spearman_rho',
                    'ci_95_lo','ci_95_hi']].to_string(index=False))
print('\nPARTIAL SPEARMAN (housing controlling for poverty):')
print(partial_df.to_string(index=False))
print('\nMODEL PERFORMANCE (temporal holdout 2021-2023):')
perf = rf_df.merge(ols_df[['outcome','ols_r2']], on='outcome')
print(perf[['outcome','n_train','n_test',
            'rf_r2','rf_rmse','ols_r2']].to_string(index=False))
print('\nKEY FINDING:')
print('Housing instability (pct_severe_housing_problems) shows weak')
print('associations with premature death (rho=0.121) and mental health')
print('days (rho=0.179), and is not significantly associated with')
print('preventable hospital stays (rho=0.005, p=0.453).')
print('Child poverty dominates across all three outcomes.')
print('After controlling for poverty, housing instability shows negative')
print('partial correlations with premature death and preventable stays.')
print('\nAll results saved to /results/')